# 01 - Prepare STATS19 Data

## Overview

The UK Department for Transport (DfT) road casualty statistics consist of [three primary datasets](https://www.data.gov.uk/dataset/cb7ae6f0-4be6-4935-9277-47e5ce24a11f/road-accidents-safety-data):
- Collision data: Information about each accident event
- Vehicle data: Details about vehicles involved in accidents
- Casualty data: Information about people injured in accidents

These datasets are linked by common identifiers and can be quite large (approximately 4.5GB combined).

## Memory Challenges

When processing large datasets, loading everything into memory at once can lead to `MemoryError` issues. A solution is to process the data in manageable chunks, focusing on:

1. Reading data incrementally 
2. Applying early filtering to reduce data volume
3. Only loading relevant portions of secondary datasets
4. Writing results incrementally to disk

In [1]:
# Import function modules
import sys
sys.path.append('../utilities')
import util.data_load_tools as dlt
from pathlib import Path

### Chunked Processing Functions
We define specialized functions to handle large data processing:
- `load_filtered_csv`: loads and filters `.csv` files in chunks.

In [6]:
help(dlt.load_filtered_csv)

Help on function load_filtered_csv in module data_load_tools:

load_filtered_csv(path, filter_func=None, dtype=None, chunksize=100000)
    Load and filter a CSV file in chunks

    Parameters:
    path : Path to CSV file
    filter_func : Function to filter rows (optional)
    dtype : Dictionary of column data types
    chunksize : Number of rows to process at once



- `filter_south_yorkshire`: filter function to isolate data for South Yorkshire

In [7]:
help(dlt.filter_south_yorkshire)

Help on function filter_south_yorkshire in module data_load_tools:

filter_south_yorkshire(df)



`clean_and_organise_data`: removes redundant columns and orders identifier columns to the front.

In [8]:
help(dlt.clean_and_organize_data)

Help on function clean_and_organize_data in module data_load_tools:

clean_and_organize_data(merged_data)
    Clean up and organize columns in the merged dataset

    Parameters:
    merged_data : DataFrame containing the merged data

    Returns:
    DataFrame with cleaned and reorganized columns



- `process_in_chunks`: Main processing function that handles the entire workflow. Keeps relationship integrity.

In [9]:
help(dlt.process_in_chunks)

Help on function process_in_chunks in module data_load_tools:

process_in_chunks(casualty_path, collision_path, vehicle_path, output_path, filter_func=None, chunksize=50000, dtype_dict=None)
    Process large datasets while preserving relationships between records

    Parameters:
    casualty_path : Path to casualty CSV
    collision_path : Path to collision CSV
    vehicle_path : Path to vehicle CSV
    output_path : Where to save the final filtered data
    filter_func : Function to filter the dataset (optional)
    chunksize : Number of rows to process at once
    dtype_dict : Dictionary of column data types



This function:

1. Processes collision data in chunks
2. For each chunk, applies filtering if specified
3. Identifies accident indices in the current chunk
4. Loads only relevant casualty and vehicle data using these indices
5. Merges the datasets appropriately
6. Cleans and organizes the columns
7. Writes each processed chunk to the output file
8. Frees memory after each chunk is processed

#### Memory Optimization Strategy
Our approach follows these key principles:

1. Early filtering: Apply geographic filtering (South Yorkshire) early to minimize data volume
2. Selective loading: Only load data relevant to the current processing chunk
3. Incremental output: Write results to disk as they're processed rather than accumulating in memory
4. Memory cleanup: Explicitly delete intermediate dataframes after they're no longer needed

#### Data Type Specification
We specify data types for identifier columns to ensure consistent joining:

In [9]:
# Specify data types for critical columns
dtype_dict = {
    'accident_index': str,
    'accident_year': str, 
    'accident_reference': str,
    'vehicle_reference': str,
    'casualty_reference': str
}

Now that we have the functions defined, we can run the code and process our datasets:

In [10]:
# File paths
my_dir_path = Path('F:/downloads')
save_path = Path('../data/datasets')
output_file = '../data/STATS19/dft_STATS19_1979_23_SY.csv'

# Process the data in chunks
process_in_chunks(
    casualty_path=my_dir_path/'dft-road-casualty-statistics-casualty-1979-latest-published-year.csv',
    collision_path=my_dir_path/'dft-road-casualty-statistics-collision-1979-latest-published-year.csv',
    vehicle_path=my_dir_path/'dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv',
    output_path=output_file,
    filter_func=filter_south_yorkshire,
    chunksize=50_000,
    dtype_dict=dtype_dict
)

Step 1: Filtering collision data to get relevant accident indices...
Found 180391 relevant accidents

Step 2: Processing casualty data...
Processed 243191 casualty records

Step 3: Processing vehicle data...
Processed 315043 vehicle records

Step 4: Loading filtered collision data...
Loaded 180391 collision records

Step 5: Merging datasets...
  Merging casualty data with collision data...
  Merging with vehicle data...

Step 6: Cleaning and organizing data...
  Cleaning and organizing columns...

Step 7: Writing processed data to file...

Processing complete. 243191 records saved to ../data/STATS19/dft_STATS19_1979_23_SY.csv


In [3]:
import pandas as pd
df = pd.read_csv('../data/STATS19/dft_STATS19_1979_23_SY.csv')

is_valid = dlt.validate_stats19_relationships(df)
print(f"Data validation: {'Passed' if is_valid else 'Failed'}")

C:\Users\REALME\AppData\Local\Temp\ipykernel_11128\3897918169.py:2: DtypeWarning: Columns (0,2,18,33,34,53,75,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/STATS19/dft_STATS19_1979_23_SY.csv')


Data validation: Passed
